# 01 — Baseline ResNet18 ((v2))

Correccion: Usa los manifiestos completos sin re-dividir.

Cambios v2:
- Dataset 4x mas grande (todas las vistas por paciente)
- Data leakage eliminado
- Usa manifest_train/val/test.csv directamente (sin re-dividir)
- Batch size 32, weight decay 1e-4
- Early stopping (patience=5)
- LR scheduler (ReduceLROnPlateau)

## Resultados esperados
- Train: 4,278 imgs (999 pac)
- Val: 1,134 imgs (249 pac)
- Test: 1,012 imgs (318 pac)
- AUC test >= 0.70


In [20]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader

from src.data import MammographyDataset
from src.data.preprocessing import get_train_transforms, get_val_transforms, SEED
from src.evaluation import compute_all_metrics, compute_youden_threshold, print_metrics_table

# Configuracion
BATCH_SIZE = 32
LR = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 20
EARLY_STOP_PATIENCE = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'resnet18_baseline_v2'
OUTPUT_DIR = '../reports/models'
EXPERIMENT_DIR = '../reports/experiments'

print('=== CONFIGURACION ===')
print(f'Device: {DEVICE}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Learning rate: {LR}')
print(f'Weight decay: {WEIGHT_DECAY}')
print(f'Epocas: {EPOCHS}')
print(f'Early stopping: {EARLY_STOP_PATIENCE}')
print(f'Seed: {SEED}')

=== CONFIGURACION ===
Device: cuda
Batch size: 32
Learning rate: 0.0001
Weight decay: 0.0001
Epocas: 20
Early stopping: 5
Seed: 42


In [21]:
# CORREGIDO: Cargar los 3 manifiestos directamente SIN re-dividir
MANIFEST_TRAIN = '../src/data/processed/manifest_train.csv'
MANIFEST_VAL = '../src/data/processed/manifest_val.csv'
MANIFEST_TEST = '../src/data/processed/manifest_test.csv'

df_train = pd.read_csv(MANIFEST_TRAIN)
df_val = pd.read_csv(MANIFEST_VAL)
df_test = pd.read_csv(MANIFEST_TEST)

print('\n=== DATOS (manifiestos completos) ===')
print(f'Train: {len(df_train)} imgs ({df_train["patient_id"].nunique()} pacientes)')
print(f'Val: {len(df_val)} imgs ({df_val["patient_id"].nunique()} pacientes)')
print(f'Test: {len(df_test)} imgs ({df_test["patient_id"].nunique()} pacientes)')
print(f'Total: {len(df_train) + len(df_val) + len(df_test)} imagenes')

print('\nDistribucion de clases:')
for name, df in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    counts = df['label'].value_counts().sort_index()
    total = len(df)
    print(f'  {name}: Benigno={counts.get(0, 0)} ({counts.get(0, 0)/total*100:.1f}%), Maligno={counts.get(1, 0)} ({counts.get(1, 0)/total*100:.1f}%)')

# Verificar data leakage
train_p = set(df_train['patient_id'])
val_p = set(df_val['patient_id'])
test_p = set(df_test['patient_id'])
print('\nVerificacion data leakage:')
print(f'  Train-Val: {len(train_p & val_p)} pacientes')
print(f'  Train-Test: {len(train_p & test_p)} pacientes')
print(f'  Val-Test: {len(val_p & test_p)} pacientes')


=== DATOS (manifiestos completos) ===
Train: 4278 imgs (999 pacientes)
Val: 1134 imgs (249 pacientes)
Test: 1012 imgs (318 pacientes)
Total: 6424 imagenes

Distribucion de clases:
  Train: Benigno=2439 (57.0%), Maligno=1839 (43.0%)
  Val: Benigno=631 (55.6%), Maligno=503 (44.4%)
  Test: Benigno=594 (58.7%), Maligno=418 (41.3%)

Verificacion data leakage:
  Train-Val: 0 pacientes
  Train-Test: 0 pacientes
  Val-Test: 0 pacientes


In [22]:
# Datasets y DataLoaders
train_ds = MammographyDataset(df_train, transform=get_train_transforms())
val_ds = MammographyDataset(df_val, transform=get_val_transforms())
test_ds = MammographyDataset(df_test, transform=get_val_transforms())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_weights = train_ds.get_class_weights()
print(f'\nPesos de clase: {class_weights}')
print(f'Batches: Train={len(train_loader)}, Val={len(val_loader)}, Test={len(test_loader)}')


Pesos de clase: tensor([0.4299, 0.5701])
Batches: Train=133, Val=36, Test=32


In [23]:
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import models
from torchvision.models import ResNet18_Weights

# Modelo
torch.manual_seed(SEED)
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(512, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

print('=== MODELO ===')
print(f'Parametros: {sum(p.numel() for p in model.parameters()):,}')
print(f'Optimizer: Adam (lr={LR}, wd={WEIGHT_DECAY})')
print(f'Scheduler: ReduceLROnPlateau')

=== MODELO ===
Parametros: 11,177,538
Optimizer: Adam (lr=0.0001, wd=0.0001)
Scheduler: ReduceLROnPlateau


In [5]:
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import time

print('\n=== ENTRENAMIENTO ===')
history = []
best_auc = 0.0
best_path = None
patience_counter = 0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    
    # Train
    model.train()
    tr_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(train_loader, desc=f'Epoca {epoch:02d}/{EPOCHS}', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)

    # Validation
    model.eval()
    vl_loss, vl_correct, vl_total = 0.0, 0, 0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, labels)
            vl_loss += loss.item() * images.size(0)
            vl_correct += (logits.argmax(1) == labels).sum().item()
            vl_total += images.size(0)
            all_probs.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    vl_auc = roc_auc_score(all_labels, all_probs)
    epoch_time = time.time() - epoch_start
    
    row = {
        'epoch': epoch,
        'tr_loss': tr_loss / total,
        'tr_acc': correct / total,
        'vl_loss': vl_loss / vl_total,
        'vl_acc': vl_correct / vl_total,
        'vl_auc': vl_auc,
        'lr': optimizer.param_groups[0]['lr']
    }
    history.append(row)
    
    print(f'Epoca {epoch:02d}/{EPOCHS} ({epoch_time:.1f}s) | '
          f'Loss {row["tr_loss"]:.4f}/{row["vl_loss"]:.4f} | '
          f'Acc {row["tr_acc"]:.4f}/{row["vl_acc"]:.4f} | '
          f'AUC {vl_auc:.4f} | LR {row["lr"]:.2e}')

    scheduler.step(vl_auc)
    
    if vl_auc > best_auc:
        best_auc = vl_auc
        patience_counter = 0
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        best_path = os.path.join(OUTPUT_DIR, f'{MODEL_NAME}.pt')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_auc': best_auc,
        }, best_path)
        print(f'  -> Checkpoint guardado (AUC={best_auc:.4f})')
    else:
        patience_counter += 1
        print(f'  -> Sin mejora ({patience_counter}/{EARLY_STOP_PATIENCE})')
    
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping en epoca {epoch}')
        break

print(f'\nTiempo total: {(time.time() - start_time)/60:.1f} min')
print(f'Mejor AUC val: {best_auc:.4f}')


=== ENTRENAMIENTO ===


Epoca 01/20 (703.1s) | Loss 0.6660/0.6727 | Acc 0.6158/0.6085 | AUC 0.6617 | LR 1.00e-04
  -> Checkpoint guardado (AUC=0.6617)


Epoca 02/20 (697.8s) | Loss 0.6277/0.6197 | Acc 0.6461/0.6631 | AUC 0.7327 | LR 1.00e-04
  -> Checkpoint guardado (AUC=0.7327)


Epoca 03/20 (773.3s) | Loss 0.5992/0.6515 | Acc 0.6715/0.6270 | AUC 0.7087 | LR 1.00e-04
  -> Sin mejora (1/5)


Epoca 04/20 (716.1s) | Loss 0.5682/0.6182 | Acc 0.7035/0.6667 | AUC 0.7302 | LR 1.00e-04
  -> Sin mejora (2/5)


Epoca 05/20 (633.8s) | Loss 0.5439/0.6232 | Acc 0.7188/0.6561 | AUC 0.7364 | LR 1.00e-04
  -> Checkpoint guardado (AUC=0.7364)


... (10 outputs left out)

Epoca 16/20 (692.1s) | Loss 0.2865/0.8004 | Acc 0.8776/0.6614 | AUC 0.7487 | LR 1.00e-04
  -> Sin mejora (2/5)


Epoca 17/20 (685.8s) | Loss 0.2401/0.9627 | Acc 0.9004/0.6596 | AUC 0.7337 | LR 1.00e-04
  -> Sin mejora (3/5)


Epoca 18/20 (653.9s) | Loss 0.2365/0.9722 | Acc 0.8999/0.6429 | AUC 0.7050 | LR 1.00e-04
  -> Sin mejora (4/5)


Epoca 19/20 (612.4s) | Loss 0.1880/0.9726 | Acc 0.9206/0.6878 | AUC 0.7461 | LR 5.00e-05
  -> Sin mejora (5/5)

Early stopping en epoca 19

Tiempo total: 206.0 min
Mejor AUC val: 0.7667


In [24]:
# Cargar el modelo guardado (para reiniciar sesion)
import torch
import os

MODEL_NAME = 'resnet18_baseline_v2'
OUTPUT_DIR = '../reports/models'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

best_path = os.path.join(OUTPUT_DIR, f'{MODEL_NAME}.pt')
print(f'Cargando checkpoint desde: {best_path}')

# Reconstruir arquitectura
from torchvision import models
import torch.nn as nn
from src.data.preprocessing import SEED

torch.manual_seed(SEED)
model = models.resnet18(weights=None)
model.fc = nn.Linear(512, 2)
model = model.to(DEVICE)

# Cargar pesos
checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Modelo cargado (epoca {checkpoint["epoch"]}, AUC val={checkpoint["best_auc"]:.4f})')

Cargando checkpoint desde: ../reports/models\resnet18_baseline_v2.pt
Modelo cargado (epoca 14, AUC val=0.7667)


In [ ]:
# Evaluar en test
checkpoint = torch.load(best_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Modelo cargado (epoca {checkpoint["epoch"]}, AUC {checkpoint["best_auc"]:.4f})')

model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        logits = model(images)
        probs = torch.softmax(logits, 1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# Calibracion del umbral en VALIDACION (sin fuga a prueba; auditoria 25-sep-2026)
val_probs, val_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        logits = model(images)
        val_probs.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
        val_labels.extend(labels.numpy())
val_probs = np.array(val_probs)
val_labels = np.array(val_labels)

# Umbral 0.5
all_preds_05 = (all_probs >= 0.5).astype(int)
metrics_05 = compute_all_metrics(all_labels, all_preds_05, y_prob=all_probs)
print('\n=== TEST (umbral 0.5) ===')
print_metrics_table(metrics_05, 'Test')

# Umbral Youden (calibrado en validacion, aplicado congelado a prueba)
best_threshold, _, _, _ = compute_youden_threshold(val_labels, val_probs)
all_preds_youden = (all_probs >= best_threshold).astype(int)
youden_metrics = compute_all_metrics(all_labels, all_preds_youden, y_prob=all_probs)
print(f'\nUmbral Youden (val): {best_threshold:.4f}')
print('\n=== TEST (umbral de validacion) ===')
print_metrics_table(youden_metrics, 'Test Youden-val')

In [ ]:
import json

results = {
    'experimento': 'ResNet18 Baseline V2',
    'config': {'batch_size': BATCH_SIZE, 'lr': LR, 'weight_decay': WEIGHT_DECAY, 'epochs': EPOCHS},
    'dataset': {'train': len(df_train), 'val': len(df_val), 'test': len(df_test)},
    'mejor_epoca': int(checkpoint['epoch']),
    'mejor_auc_val': float(checkpoint['best_auc']),
    'auc_test': float(youden_metrics['roc_auc']),
    'recall_test': float(youden_metrics['recall']),
    'umbral_youden': float(best_threshold),
    'nota_calibracion': 'Umbral de Youden calibrado en validacion y aplicado congelado a prueba (auditoria 25-sep-2026).',
    'historial': history if 'history' in dir() else 'no disponible (sesion reiniciada)'
}

os.makedirs(EXPERIMENT_DIR, exist_ok=True)
with open(os.path.join(EXPERIMENT_DIR, '01_resnet18_baseline_v2.json'), 'w') as f:
    json.dump(results, f, indent=2, default=str)

print('\n=== RESUMEN ===')
print(f'AUC val: {checkpoint["best_auc"]:.4f}')
print(f'AUC test: {youden_metrics["roc_auc"]:.4f}')
print(f'Recall test: {youden_metrics["recall"]:.4f}')
print('\nGuardado en reports/experiments/01_resnet18_baseline_v2.json')